Another proof of concept project to do the following objectives: 
1. Generate RSA public/private keys and establish a secure session key between two enterprise systems. 
2. Securely encrypt, exchange, and validate the shared session key for trusted communication.
3. Simulate encrypted infrastructure communication and authorized payload recovery across systems.

In [1]:
from cryptography.fernet import Fernet
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes
from tabulate import tabulate
from pathlib import Path
import hashlib

print ("\n===========================================================")
print (" Secure Communication Between Two Enterprise Systems")
print ("\n===========================================================")


 Secure Communication Between Two Enterprise Systems



In [2]:
# ================================
# 1. Create shared workspace
# ================================

workspace = Path("secure_sync_demo")
workspace.mkdir(exist_ok=True)

print(f"\nSecure communication workspace initialized")


Secure communication workspace initialized


In [3]:
# ============================================================
# 2. Remote Backup System prepares deployment package
# ============================================================

deployment_file = workspace / "deployment_instructions.txt"

deployment_content = """
SECURE INFRASTRUCTURE DEPLOYMENT PLAN
-------------------------------
Primary Region: US-East
Backup Region: Europe-West
Disaster Recovery Sync: Enabled
Internal Classification: Restricted
"""

deployment_file.write_text(deployment_content)

print("\nDeployment instructions created:")
print(deployment_content)


Deployment instructions created:

SECURE INFRASTRUCTURE DEPLOYMENT PLAN
-------------------------------
Primary Region: US-East
Backup Region: Europe-West
Disaster Recovery Sync: Enabled
Internal Classification: Restricted



In [4]:
# ==========================================================
# 3. Headquarters Server creates RSA identity
# ==========================================================

print("\n===============================================")
print("[Headquarters Infrastructure Server]")
print("Generating secure RSA identity")
print("===============================================")


server_private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048
)

server_public_key = server_private_key.public_key()

print("RSA 2048-bit identity established")


[Headquarters Infrastructure Server]
Generating secure RSA identity
RSA 2048-bit identity established


In [5]:
# =============================================
# 4. Remote Backup System generates session key
# =============================================

print("\n===============================================")
print("[Remote Backup System]")
print("Generating temporary secure session key")
print("===============================================")

session_key = Fernet.generate_key()

print("\nSession key created:")
print(session_key.decode())


[Remote Backup System]
Generating temporary secure session key

Session key created:
E4iEwv4uZxuUMlH1pQQ74FzcwOXwMFwdetia_oXIzwg=


In [6]:
# ============================================================
# 5. Remote system protects session key using RSA
# ============================================================

print("\nEncrypting session key using headquarters public key...")

encrypted_session_key = server_public_key.encrypt(
    session_key,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

session_key_file = workspace / "protected_session.key"

session_key_file.write_bytes(encrypted_session_key)

print("Protected session key securely transferred")


Encrypting session key using headquarters public key...
Protected session key securely transferred


In [7]:
# ==========================================================
# 6. Headquarters server recovers session key
# ==========================================================

print("\n===============================================")
print("[Headquarters Infrastructure Server]")
print("Recovering protected session key...")
print("===============================================")

recovered_session_key = server_private_key.decrypt(
    encrypted_session_key,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)


print("\nSecure session established successfully")


[Headquarters Infrastructure Server]
Recovering protected session key...

Secure session established successfully


In [8]:
# =============================================
# 7. Remote system encrypts deployment payload
# =============================================

print("\n===============================================")
print("[Remote Backup System]")
print("Encrypting deployment instructions...")
print("===============================================")

secure_cipher = Fernet(session_key)

original_data = deployment_file.read_bytes()

encrypted_payload = secure_cipher.encrypt(original_data)

encrypted_payload_file = workspace / "deployment_payload.encrypted"

encrypted_payload_file.write_bytes(encrypted_payload)

print("\nEncrypted deployment payload generated")


[Remote Backup System]
Encrypting deployment instructions...

Encrypted deployment payload generated


In [ ]:
# =================================================
# 8. Unauthorized system attempts recovery
# =================================================

print("\nAttempting decryption with incorrect AES key...")

unauthorized_key = Fernet.generate_key()

unauthorized_cipher = Fernet(unauthorized_key)


try:
    unauthorized_cipher.decrypt(encrypted_payload)

    print("Unexpected unauthorized recovery success")

except Exception:

    print("\nUnauthorized recovery failed")
    print("Incorrect session key rejected")


Attempting decryption with incorrect AES key...

Unauthorized recovery Failed
Incorrect session key rejected


In [10]:
# =================================================
# 9. Headquarters server decrypts payload
# =================================================

print("\n===============================================")
print("[Headquarters Infrastructure Server]")
print("Decrypting authorized deployment payload...")
print("===============================================")

authorized_cipher = Fernet(recovered_session_key)

decrypted_payload = authorized_cipher.decrypt(
    encrypted_payload
)

recovered_file = workspace / "deployment_payload_recovered.txt"

recovered_file.write_bytes(decrypted_payload)

print("\nAuthorized payload recovery successful")

print("\n===Recovered deployment instructions")
print(decrypted_payload.decode())


[Headquarters Infrastructure Server]
Decrypting authorized deployment payload...

Authorized payload recovery successful

===Recovered deployment instructions

SECURE INFRASTRUCTURE DEPLOYMENT PLAN
-------------------------------
Primary Region: US-East
Backup Region: Europe-West
Disaster Recovery Sync: Enabled
Internal Classification: Restricted



In [11]:
# ===================================
# 10. Validate communication integrity
# ===================================

original_hash = hashlib.sha256(original_data).hexdigest()

recovered_hash = hashlib.sha256(decrypted_payload).hexdigest()

validation = [
    ["Original Payload Hash", original_hash],
    ["Recovered Payload Hash", recovered_hash]
]

print("\n=== Integrity Validation ===")

print(tabulate(
    validation,
    headers=["Artifact", "SHA-256 Hash"],
    tablefmt="grid"
))


=== Integrity Validation ===
+------------------------+------------------------------------------------------------------+
| Artifact               | SHA-256 Hash                                                     |
+========================+==================================================================+
| Original Payload Hash  | 5c419269e034a4e6537acb2dd25d82d7c0c2525516e90aaf22ed241aba30c8fc |
+------------------------+------------------------------------------------------------------+
| Recovered Payload Hash | 5c419269e034a4e6537acb2dd25d82d7c0c2525516e90aaf22ed241aba30c8fc |
+------------------------+------------------------------------------------------------------+
